### Is the Admission and Discharge Pathway linked to 30-Day Readmission?

In [2]:
import pandas as pd

# Load the cleaned data made in notebook 01
df = pd.read_csv('../datasets/diabetic_clean.csv')

# Checkpoint: expect 69,982 rows and 50 columns
print(df.shape)

# readmitted_30d is 1 (readmitted within 30 days) or 0 (not),
# so its mean is the share of patients readmitted
print('Overall 30-day readmission rate:', round(df['readmitted_30d'].mean() * 100, 2), '%')

(69982, 50)
Overall 30-day readmission rate: 8.98 %


In [4]:
# ---------- 1) Discharge destination (where the patient went after leaving) ----------

discharge_names = {
    1:  'Home',                               2:  'Another short-term hospital',
    3:  'Skilled nursing facility',           4:  'Intermediate care facility',
    5:  'Other inpatient institution',        6:  'Home with home health',
    7:  'Left against medical advice',        8:  'Home with IV provider',
    9:  'Admitted to this hospital',          10: 'Neonate to another hospital',
    12: 'Still patient / return for outpatient',
    15: 'Swing bed (same hospital)',          16: 'Referred for outpatient elsewhere',
    17: 'Referred for outpatient here',       18: 'Blank in mapping file',
    22: 'Rehab facility',                     23: 'Long-term care hospital',
    24: 'Medicaid-only nursing facility',     25: 'Not mapped',
    27: 'Federal health facility',            28: 'Psychiatric hospital',
}
discharge_group = df['discharge_disposition_id'].map(discharge_names)

print('Discharge codes with no name:', discharge_group.isna().sum())

Discharge codes with no name: 0


In [7]:
# ---------- 2) Admission type (how urgent the admission was) ----------
admission_type_names = {
    1: 'Emergency', 2: 'Urgent', 3: 'Elective', 4: 'Newborn',
    5: 'Not available', 6: 'Blank in mapping file', 7: 'Trauma center', 8: 'Not mapped',
}
admission_type_group = df['admission_type_id'].map(admission_type_names)

print('Admission type codes with no name:', admission_type_group.isna().sum())

table_adm = df.groupby(admission_type_group)['readmitted_30d'].agg(['count', 'mean'])
table_adm['mean'] = (table_adm['mean'] * 100).round(2)
table_adm = table_adm.sort_values('count', ascending=False)  

print('ADMISSION TYPE: patients and 30-day readmission % (ignore the 2 tiny groups)')
print(table_adm)

Admission type codes with no name: 0
ADMISSION TYPE: patients and 30-day readmission % (ignore the 2 tiny groups)
                       count   mean
admission_type_id                  
Emergency              35477   9.18
Elective               13783   8.29
Urgent                 12803   8.97
Blank in mapping file   4515   9.97
Not available           3086   8.55
Not mapped               291   7.22
Trauma center             18   0.00
Newborn                    9  11.11


In [8]:
# ---------- 3) Admission source (where the patient came from) ----------
source_names = {
    7:  'Emergency room',
    1:  'Physician referral',
    17: 'Blank in mapping file',
    4:  'Transfer from a hospital',
    6:  'Transfer from another health facility',
    2:  'Clinic referral',
    5:  'Transfer from a skilled nursing facility',
}
table_src = df.groupby('admission_source_id')['readmitted_30d'].agg(['count', 'mean'])
table_src['mean'] = (table_src['mean'] * 100).round(2)
table_src = table_src.sort_values('count', ascending=False)

table_src = table_src[table_src['count'] >= 400].rename(index=source_names)

print('ADMISSION SOURCE: patients and 30-day readmission %')
print(table_src)
print('Patients covered:', table_src['count'].sum(), '(expect 69,570)')

ADMISSION SOURCE: patients and 30-day readmission %
                                          count  mean
admission_source_id                                  
Emergency room                            37272  9.26
Physician referral                        21743  8.61
Blank in mapping file                      4821  8.75
Transfer from a hospital                   2530  8.54
Transfer from another health facility      1785  8.07
Clinic referral                             908  8.92
Transfer from a skilled nursing facility    511  9.39
Patients covered: 69570 (expect 69,570)
